# BIPV Facade Analysis - Colab Runner

Use this notebook in Google Colab with a GPU runtime. The project logic lives in `src/`; this notebook only installs dependencies, mounts Drive, sets inputs, and runs the pipeline.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

After this cell finishes, Colab may ask you to restart the runtime. If it does, restart and continue from the next cell.

In [ ]:
!pip install -q -r requirements-colab.txt
print('Dependencies installed.')

## 3. Configure Input

In [ ]:
IMAGE_PATH = '/content/drive/MyDrive/BIPV_images/Build5.jpeg'
OUTPUT_PATH = '/content/drive/MyDrive/BIPV_images/pvsyst_export.json'

# Optional Google Earth measurements. Leave as None if unavailable.
GE_WIDTH_M = None
GE_HEIGHT_M = None

# Optional manual floor count override. Leave as None for automatic detection.
KNOWN_FLOORS = None

# Keep the rectified image at the original H x W so visual size does not drift.
PRESERVE_ORIGINAL_SIZE = True

# If DINO misses windows, SAM fallback searches inside the facade mask.
MIN_WINDOW_DETECTIONS = 3

## 4. Run Pipeline

In [ ]:
from src.config import AnalysisConfig
from src.pipeline import run_bipv_analysis

config = AnalysisConfig(
    image_path=IMAGE_PATH,
    output_path=OUTPUT_PATH,
    ge_width_m=GE_WIDTH_M,
    ge_height_m=GE_HEIGHT_M,
    known_floors=KNOWN_FLOORS,
    run_stable_diffusion=True,
    preserve_original_size=PRESERVE_ORIGINAL_SIZE,
    min_window_detections=MIN_WINDOW_DETECTIONS,
)

result = run_bipv_analysis(config)
print(f"PVsyst export saved to: {result['output_path']}")

## 5. View Key Outputs

In [ ]:
from src.visualization import show_image, show_mask_overlay, show_side_by_side

show_side_by_side(result['image_rgb'], result['clean_image'], 'Original', 'Obstacle Removed')
show_image(result['aligned_facade'], 'Rectified Facade')
show_mask_overlay(result['aligned_facade'], result['usable_results']['usable_mask'], color=(0, 255, 0), title='Usable BIPV Area')

print('Dimensions:', result['dimensions'])
print('Panel capacity:', result['panel_capacity'])